# Text Complexity Combo Evaluator (Early Release)

This evaluator combines four distinct evaluators: the **Sentence Structure Evaluator**, the **Vocabulary Evaluator**, the **Subject Matter Knowledge (SMK) Evaluator**, and the **Conventionality Evaluator**. It is designed to assess the complexity of texts for educational purposes, particularly in relation to grade-level expectations.

**The Sentence Structure Evaluator** analyzes the syntactic complexity of a passage, helping to identify whether sentence constructions are appropriate for the intended grade level.

**The Vocabulary Evaluator** gives developers the fine-grained insight they need but can't get from traditional tools. It helps determine whether texts use words that align with grade-level expectations and support growth in academic language. This ensures students are consistently exposed to the kinds of vocabulary that build knowledge and enable them to fully engage with grade-level texts.

**The Subject Matter Knowledge (SMK) Evaluator** assesses the background knowledge demands of informational text. It returns a structured output that identifies the core subjects and concepts present in the text, and determines the extent of background knowledge required to comprehend it.

**The Conventionality Evaluator** measures how explicit, literal, and straightforward a text's meaning is. It assesses whether meaning is on the surface or requires abstract reasoning, interpretation of figurative language, or familiarity with unconventional expressions.

### Install & Load necessary packages

In [1]:
%pip install -qU pydantic textstat langchain langchain_openai langchain-google-genai

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Load packages
import ast
import asyncio
import getpass
import json
import os
from typing import Any, List, Literal

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.chat import HumanMessagePromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from textstat import textstat as ts

### Set up the evaluator's model and prompts

In [3]:
import sys
from pathlib import Path

# Each evaluator's prompt module now lives in its own dimension subfolder
# (e.g. vocabulary/prompts/vocab_prompts.py). The dimension folder names
# contain hyphens, so they can't be imported as packages. Instead we add
# each prompts/ directory to sys.path and import the modules by filename.
_QTC_ROOT = Path.cwd()
for _dim in (
    "vocabulary",
    "sentence-structure",
    "subject-matter-knowledge",
    "conventionality",
):
    sys.path.append(str(_QTC_ROOT / _dim / "prompts"))

import vocab_prompts as v_prompts
import sent_str_prompts as s_prompts
import smk_prompts as smk_prompts
import conventionality_prompts as conv_prompts

# Set your api keys in your environment, .env file, or enter when prompted.
# os.environ['GOOGLE_API_KEY'] = 'YOUR API KEY'
# os.environ['OPENAI_API_KEY'] = 'YOUR API KEY'
load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API key: ")

# Define the models to be used for vocabulary complexity.
# Grades 3-4 were validated with Gemini; grades 5-12 with GPT-4.1
# (matching the standalone vocabulary evaluator's grade-specific prompts).
VOCAB_TEMPERATURE = 0
VOCAB_MODEL_GRADES_3_4 = "gemini-2.5-pro"
vocab_complexity_model_grades_3_4 = ChatGoogleGenerativeAI(
    model=VOCAB_MODEL_GRADES_3_4, temperature=VOCAB_TEMPERATURE
)
VOCAB_MODEL_OTHER_GRADES = "gpt-4.1"
vocab_complexity_model_other_grades = ChatOpenAI(
    model=VOCAB_MODEL_OTHER_GRADES, temperature=VOCAB_TEMPERATURE
)

# Define the model to be used for student background knowledge generation and sentence structure complexity
COMBO_MODEL = "gpt-4o-2024-11-20"
COMBO_TEMPERATURE = 0
model = ChatOpenAI(model=COMBO_MODEL, temperature=COMBO_TEMPERATURE)

# Define the model to be used for SMK complexity
SMK_MODEL = "gemini-3-flash-preview"
SMK_TEMPERATURE = 0
smk_model = ChatGoogleGenerativeAI(model=SMK_MODEL, temperature=SMK_TEMPERATURE)

# Define the model to be used for conventionality complexity
CONV_MODEL = "gemini-3-flash-preview"
CONV_TEMPERATURE = 0
conv_model = ChatGoogleGenerativeAI(model=CONV_MODEL, temperature=CONV_TEMPERATURE)

### Set up student background knowledge generator

In [4]:
async def get_background_knowledge_assumption(text, grade):
    """Use the background knowledge prompt from the prompts file."""
    prompt = v_prompts.bk_prompt.format(text=text, grade=grade)

    return (await model.ainvoke(prompt)).content

### Set up the input variables and output format

In [5]:
class VocabOutput(BaseModel):
    tier_2_words: str = Field(description="List of Tier 2 words")
    tier_3_words: str = Field(description="List of Tier 3 words")
    archaic_words: str = Field(description="List of Archaic words")
    other_complex_words: str = Field(description="List of Other Complex words")
    complexity_score: str = Field(
        description="the complexity of the text, one of: slightly complex, moderately complex, very complex, or exceedingly complex"
    )
    reasoning: str = Field(description="your reasoning for your answer")


prompt_vars = {
    "inputVars": [
        "text",
        "student_grade_level",
        "student_background_knowledge",
        "fk_level",
    ],
    "outputParser": JsonOutputParser(pydantic_object=VocabOutput),
}


class SmkOutput(BaseModel):
    identified_topics: List[str] = Field(
        description="List of major subjects/concepts found in the text."
    )
    curriculum_check: str = Field(
        description="Analysis of whether these topics are new or review based on the Grade Level and Reference list."
    )
    assumptions_and_scaffolding: str = Field(
        description="Analysis of what the author assumes vs. what is explained (and if definitions are provided)."
    )
    friction_analysis: str = Field(
        description="Explicit statement distinguishing if difficulty comes from Vocabulary/Structure or actual Knowledge."
    )
    complexity_score: Literal[
        "slightly_complex",
        "moderately_complex",
        "very_complex",
        "exceedingly_complex",
    ] = Field(description="The subject matter knowledge complexity level of the text")
    reasoning: str = Field(
        description="A brief synthesis of why the text fits the chosen complexity level."
    )


smk_prompt_vars = {
    "inputVars": ["text", "grade", "fk_score"],
    "outputParser": JsonOutputParser(pydantic_object=SmkOutput),
}

class ConventionalityOutput(BaseModel):
    conventionality_features: List[str] = Field(
        description="List of the specific language features driving the complexity (e.g., idioms, metaphors, implied meaning) with direct quotes from the text."
    )
    grade_context: str = Field(
        description="How the conventionality demands compare to general expectations for the provided target grade."
    )
    instructional_insights: str = Field(
        description="Actionable pedagogical suggestions for scaffolding the unconventional language features in the classroom."
    )
    complexity_score: Literal[
        "slightly_complex",
        "moderately_complex",
        "very_complex",
        "exceedingly_complex",
    ] = Field(description="The conventionality complexity level of the text")
    reasoning: str = Field(
        description="A synthesis of why the text fits the chosen rubric level."
    )


conv_prompt_vars = {
    "inputVars": ["text", "grade", "fk_score"],
    "outputParser": JsonOutputParser(pydantic_object=ConventionalityOutput),
}

In [6]:
class SentenceAnalysesEvaluatorOutput(BaseModel):
    reasoning: str = Field(
        description="Your step-by-step reasoning as you analyze the text, with a line break between each sentence analyzed."
    )

    # Foundational
    num_sentences: int = Field(description="Total number of sentences in the text.")
    num_words: int = Field(description="Total number of words in the text.")
    flesch_kincaid_grade: float = Field(
        description="Flesch-Kincaid Grade Level number from Computational Counts, rounded to two decimal places."
    )

    # Sentence Type
    num_simple_sentences: int = Field(
        description="Number of simple sentences in the text."
    )
    num_compound_sentences: int = Field(
        description="Number of compound sentences in the text."
    )
    num_complex_sentences: int = Field(
        description="Number of complex sentences in the text."
    )
    num_compound_complex_sentences: int = Field(
        description="Number of compound-complex sentences in the text."
    )
    num_other_sentences: int = Field(
        description="Number of sentences that do not fit the four canonical types (e.g., fragments, run-ons, elliptical answers, headlines, or stylized dialogue tags)."
    )

    # Subordination
    num_independent_clauses: int = Field(
        description="Total count of all independent clauses in the text."
    )
    num_subordinate_clauses: int = Field(
        description="Total count of all subordinate clauses in the text. A single sentence can have multiple."
    )
    num_total_clauses: int = Field(
        description="The sum of all independent and subordinate clauses in the text."
    )
    num_sentences_with_subordinate: int = Field(
        description="Number of sentences that contain a subordinate clause."
    )
    num_sentences_with_multiple_subordinates: int = Field(
        description="Number of sentences that contain two or more subordinate clauses."
    )
    num_sentences_with_embedded_clauses: int = Field(
        description="Number of sentences with an embedded clause (a subordinate clause inside another clause)."
    )

    # Informational Phrases
    num_prepositional_phrases: int = Field(
        description="Number of prepositional phrases in the text."
    )
    num_participle_phrases: int = Field(
        description="Number of participle phrases in the text."
    )
    num_appositive_phrases: int = Field(
        description="Number of appositive phrases in the text."
    )

    # Cohesion
    num_simple_transitions: int = Field(
        description="Number of simple transitions in the text."
    )
    num_sophisticated_transitions: int = Field(
        description="Number of sophisticated transitions in the text."
    )

    # Sentence Type Density
    words_in_simple_sentences: int = Field(
        description="Total number of words in all sentences classified as simple."
    )
    words_in_compound_sentences: int = Field(
        description="Total number of words in all sentences classified as compound."
    )
    words_in_complex_sentences: int = Field(
        description="Total number of words in all sentences classified as complex."
    )
    words_in_compound_complex_sentences: int = Field(
        description="Total number of words in all sentences classified as compound-complex."
    )
    words_in_other_sentences: int = Field(
        description="Total number of words in all sentences classified as other."
    )

    # Additional Features
    sentence_word_counts: List[int] = Field(
        description="A list containing the word count of each individual sentence in the order they appear."
    )
    num_one_concept_sentences: int = Field(
        description="Number of sentences with a single main idea: no subordinate clause and no transition word/phrase."
    )
    num_multi_concept_sentences: int = Field(
        description="Number of sentences with multiple main ideas: either a subordinate clause or a transition word/phrase or both."
    )
    num_cleft_sentences: int = Field(
        description='Number of sentences with cleft constructions (e.g., "It was X that...", "What X did was...").'
    )
    max_clauses_in_any_sentence: int = Field(
        description="Max number of clauses (independent + subordinate) found in a single sentence."
    )


class ComplexityClassificationOutput(BaseModel):
    reasoning: str = Field(
        description="Detailed reasoning that is pedagogically approriate, and helpful for K-12 educators."
    )
    answer: str = Field(
        description="The final complexity category, which must be one of ['Slightly Complex', 'Moderately Complex', 'Very Complex', 'Exceedingly Complex']."
    )

### Helper functions

In [7]:
def calculate_fk_score(text) -> float:
    """
    Calculate the Flesch-Kincaid Grade Level
    """
    fk_score = round(ts.flesch_kincaid_grade(text), 2)

    return fk_score

In [8]:
async def prepare_text_for_complexity_prediction(text, grade):
    """
    Enrich the text and grade given by user with additional features for complexity prediction.
    """
    dataset = {
        "text": text,
        "student_grade_level": grade,
        "fk_level": calculate_fk_score(text),
        "student_background_knowledge": await get_background_knowledge_assumption(
            text, grade
        ),
    }

    return dataset

In [9]:
async def execute_sentence_analysis(text: str) -> dict:

    # Compute ground truth counts
    gt_sentence_count = ts.sentence_count(text)
    gt_word_count = ts.lexicon_count(text, removepunct=True)
    gt_char_count = ts.char_count(text, ignore_spaces=True)
    gt_syllable_count = ts.syllable_count(text)
    flesch_kincaid_grade = round(ts.flesch_kincaid_grade(text), 2)

    gt_counts_str = (
        f"num_sentences: {gt_sentence_count}\n"
        f"num_words: {gt_word_count}\n"
        f"num_char: {gt_char_count}\n"
        f"num_syllable: {gt_syllable_count}\n"
        f"flesch_kincaid_grade: {flesch_kincaid_grade}"
    )

    prompt_template = ChatPromptTemplate(
        messages=[
            SystemMessage(content=s_prompts.SYSTEM_PROMPT_ANALYSIS),
            HumanMessagePromptTemplate.from_template(s_prompts.USER_PROMPT_ANALYSIS),
        ],
        input_variables=["text", "ground_truth_counts"],
        partial_variables={
            "format_instructions": JsonOutputParser(
                pydantic_object=SentenceAnalysesEvaluatorOutput
            ).get_format_instructions()
        },
    )

    chain = prompt_template | model | JsonOutputParser()
    result = await chain.ainvoke({"text": text, "ground_truth_counts": gt_counts_str})
    return result

In [10]:
FEATURE_COLS = [
    # Foundational & Distributional
    "avg_words_per_sentence",
    "sentence_length_variation",
    "percent_short_sentences",
    "percent_medium_sentences",
    "percent_long_sentences",
    "percent_very_long_sentences",
    "flesch_kincaid_grade",
    # Sentence Structure (Grammatical Type)
    "percent_simple_sentences",
    "percent_compound_sentences",
    "percent_complex_sentences",
    "percent_compound_complex_sentences",
    "percent_other_sentences",
    # Word Distribution
    "percent_words_in_simple_sentences",
    "percent_words_in_complex_sentences",
    "percent_words_in_compound_sentences",
    "percent_words_in_compound_complex_sentences",
    "percent_words_in_other_sentences",
    # Clausal & Subordination
    "avg_subordinates_per_sentence",
    "avg_clauses_per_sentence",
    "percent_sentences_with_subordinate",
    "percent_sentences_with_multiple_subordinates",
    "percent_sentences_with_embedded_clauses",
    # Phrase Density
    "prep_phrase_density",
    "participle_phrase_density",
    "appositive_phrase_density",
    # Cohesion & Transitions
    "avg_transitions_per_sentence",
    "percent_sophisticated_transitions",
    # Conceptual & Other
    "percent_sentences_w_one_concept",
    "percent_sentences_w_multi_concept",
    "percent_cleft_sentences",
    "max_clauses_in_any_sentence",
]

In [11]:
def safe_literal_eval(s):
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError, TypeError):
        # Return an empty list if parsing fails, which is a safe default for calculations
        return []


def safe_division(numerator, denominator):
    # Replaces 0 in the denominator with NaN to avoid division by zero errors
    denominator_safe = denominator.replace(0, np.nan)
    # Perform division and fill any resulting NaN values (from 0 denominators) with 0
    return (numerator / denominator_safe).fillna(0)


def categorize_sentence_lengths(word_counts):
    # This function processes a list of word counts for a single text
    if not isinstance(word_counts, list) or not word_counts:
        return pd.Series(
            [0, 0, 0, 0],
            index=[
                "percent_short_sentences",
                "percent_medium_sentences",
                "percent_long_sentences",
                "percent_very_long_sentences",
            ],
        )

    short_count, medium_count, long_count, very_long_count = 0, 0, 0, 0

    for count in word_counts:
        if count <= 10:
            short_count += 1
        elif count <= 20:
            medium_count += 1
        elif count <= 30:
            long_count += 1
        else:
            very_long_count += 1

    total_sentences = len(word_counts)

    return pd.Series(
        [
            (short_count / total_sentences) * 100,
            (medium_count / total_sentences) * 100,
            (long_count / total_sentences) * 100,
            (very_long_count / total_sentences) * 100,
        ],
        index=[
            "percent_short_sentences",
            "percent_medium_sentences",
            "percent_long_sentences",
            "percent_very_long_sentences",
        ],
    )


def add_engineered_features(df):
    df_normalized = df.copy()

    # Ensure all relevant numeric columns are actually numeric, coercing errors
    for col in df_normalized.columns:
        if col.startswith("num_") or col.startswith("words_in_"):
            df_normalized[col] = pd.to_numeric(df_normalized[col], errors="coerce")

    # Safely convert string representation of lists to actual lists
    if (
        "sentence_word_counts" in df_normalized.columns
        and not df_normalized["sentence_word_counts"].empty
    ):
        # Check if the first non-null element is a string to decide if conversion is needed
        first_item = (
            df_normalized["sentence_word_counts"].dropna().iloc[0]
            if not df_normalized["sentence_word_counts"].dropna().empty
            else None
        )
        if isinstance(first_item, str):
            df_normalized["sentence_word_counts"] = df_normalized[
                "sentence_word_counts"
            ].apply(safe_literal_eval)

    # Foundational Metrics
    df_normalized["avg_words_per_sentence"] = safe_division(
        df_normalized["num_words"], df_normalized["num_sentences"]
    )
    df_normalized["sentence_length_variation"] = df_normalized[
        "sentence_word_counts"
    ].apply(lambda x: np.std(x) if isinstance(x, list) and len(x) > 1 else 0)
    length_dist_df = df_normalized["sentence_word_counts"].apply(
        categorize_sentence_lengths
    )
    df_normalized = df_normalized.join(length_dist_df)

    # Sentence Structure Percentages
    df_normalized["percent_simple_sentences"] = (
        safe_division(
            df_normalized["num_simple_sentences"], df_normalized["num_sentences"]
        )
        * 100
    )
    df_normalized["percent_compound_sentences"] = (
        safe_division(
            df_normalized["num_compound_sentences"], df_normalized["num_sentences"]
        )
        * 100
    )
    df_normalized["percent_complex_sentences"] = (
        safe_division(
            df_normalized["num_complex_sentences"], df_normalized["num_sentences"]
        )
        * 100
    )
    df_normalized["percent_compound_complex_sentences"] = (
        safe_division(
            df_normalized["num_compound_complex_sentences"],
            df_normalized["num_sentences"],
        )
        * 100
    )
    df_normalized["percent_other_sentences"] = (
        safe_division(
            df_normalized["num_other_sentences"], df_normalized["num_sentences"]
        )
        * 100
    )

    # Word Distribution Percentages (as a percentage of total words)
    df_normalized["percent_words_in_simple_sentences"] = (
        safe_division(
            df_normalized["words_in_simple_sentences"], df_normalized["num_words"]
        )
        * 100
    )
    df_normalized["percent_words_in_compound_sentences"] = (
        safe_division(
            df_normalized["words_in_compound_sentences"], df_normalized["num_words"]
        )
        * 100
    )
    df_normalized["percent_words_in_complex_sentences"] = (
        safe_division(
            df_normalized["words_in_complex_sentences"], df_normalized["num_words"]
        )
        * 100
    )
    df_normalized["percent_words_in_compound_complex_sentences"] = (
        safe_division(
            df_normalized["words_in_compound_complex_sentences"],
            df_normalized["num_words"],
        )
        * 100
    )
    df_normalized["percent_words_in_other_sentences"] = (
        safe_division(
            df_normalized["words_in_other_sentences"], df_normalized["num_words"]
        )
        * 100
    )

    # Subordination and Clausal Complexity
    df_normalized["avg_subordinates_per_sentence"] = safe_division(
        df_normalized["num_subordinate_clauses"], df_normalized["num_sentences"]
    )
    df_normalized["avg_clauses_per_sentence"] = safe_division(
        df_normalized["num_total_clauses"], df_normalized["num_sentences"]
    )
    df_normalized["percent_sentences_with_subordinate"] = (
        safe_division(
            df_normalized["num_sentences_with_subordinate"],
            df_normalized["num_sentences"],
        )
        * 100
    )
    df_normalized["percent_sentences_with_multiple_subordinates"] = (
        safe_division(
            df_normalized["num_sentences_with_multiple_subordinates"],
            df_normalized["num_sentences"],
        )
        * 100
    )
    df_normalized["percent_sentences_with_embedded_clauses"] = (
        safe_division(
            df_normalized["num_sentences_with_embedded_clauses"],
            df_normalized["num_sentences"],
        )
        * 100
    )

    # Phrase Density (per 100 words)
    df_normalized["prep_phrase_density"] = (
        safe_division(
            df_normalized["num_prepositional_phrases"], df_normalized["num_words"]
        )
        * 100
    )
    df_normalized["participle_phrase_density"] = (
        safe_division(
            df_normalized["num_participle_phrases"], df_normalized["num_words"]
        )
        * 100
    )
    df_normalized["appositive_phrase_density"] = (
        safe_division(
            df_normalized["num_appositive_phrases"], df_normalized["num_words"]
        )
        * 100
    )

    # Cohesion and Transitions
    total_transitions = df_normalized["num_simple_transitions"].add(
        df_normalized["num_sophisticated_transitions"], fill_value=0
    )
    df_normalized["avg_transitions_per_sentence"] = safe_division(
        total_transitions, df_normalized["num_sentences"]
    )
    df_normalized["percent_sophisticated_transitions"] = (
        safe_division(df_normalized["num_sophisticated_transitions"], total_transitions)
        * 100
    )

    # Conceptual & Other
    df_normalized["percent_sentences_w_one_concept"] = (
        safe_division(
            df_normalized["num_one_concept_sentences"], df_normalized["num_sentences"]
        )
        * 100
    )
    df_normalized["percent_sentences_w_multi_concept"] = (
        safe_division(
            df_normalized["num_multi_concept_sentences"], df_normalized["num_sentences"]
        )
        * 100
    )
    df_normalized["percent_cleft_sentences"] = (
        safe_division(
            df_normalized["num_cleft_sentences"], df_normalized["num_sentences"]
        )
        * 100
    )

    return df_normalized


def normalize_label(s: Any) -> str | None:
    if s is None:
        return None
    m = {
        "slightly complex": "Slightly Complex",
        "moderately complex": "Moderately Complex",
        "very complex": "Very Complex",
        "exceedingly complex": "Exceedingly Complex",
        "extremely complex": "Exceedingly Complex",
    }
    return m.get(str(s).strip().lower())


def row_to_features_json(
    row: pd.Series, decimals: int = 1, cast_to_int: bool = True
) -> str:
    s = row.reindex(FEATURE_COLS)
    s_rounded = pd.to_numeric(s, errors="coerce").round(decimals)

    payload = (
        {k: (None if pd.isna(v) else int(v)) for k, v in s_rounded.items()}
        if cast_to_int
        else {k: (None if pd.isna(v) else float(v)) for k, v in s_rounded.items()}
    )
    return json.dumps(payload, indent=2)

In [12]:
async def classify_complexity_with_grade_level(sentence_features, grade, excerpt):

    rubric = s_prompts.GRADE_SPECIFIC_RUBRICS.get(
        grade,
        "No specific rubric available for this grade. Use general linguistic principles.",
    )

    prompt_template = ChatPromptTemplate(
        messages=[
            SystemMessage(content=s_prompts.SYSTEM_PROMPT_COMPLEXITY),
            HumanMessagePromptTemplate.from_template(s_prompts.USER_PROMPT_COMPLEXITY),
        ],
        input_variables=["sentence_features", "grade", "rubric", "excerpt"],
        partial_variables={
            "format_instructions": JsonOutputParser(
                pydantic_object=ComplexityClassificationOutput
            ).get_format_instructions()
        },
    )
    chain = prompt_template | model | JsonOutputParser()
    return await chain.ainvoke(
        {
            "sentence_features": sentence_features,
            "grade": grade,
            "rubric": rubric,
            "excerpt": excerpt,
        }
    )

In [13]:
async def evaluate_df(df: pd.DataFrame, concurrency: int = 5):
    sem = asyncio.Semaphore(concurrency)

    async def _classify_one_row(idx: Any, row: pd.Series):
        async with sem:
            try:
                feats = row_to_features_json(row)
                res = await classify_complexity_with_grade_level(
                    sentence_features=feats, grade=row["grade"], excerpt=row["text"]
                )
                return {
                    "index": idx,
                    "complexity_answer": normalize_label(res.get("answer")),
                    "complexity_reasoning": res.get("reasoning"),
                    "error": None,
                }
            except Exception as e:
                return {
                    "index": idx,
                    "complexity_answer": None,
                    "complexity_reasoning": None,
                    "error": str(e),
                }

    tasks = [_classify_one_row(idx, row) for idx, row in df.iterrows()]
    results = await asyncio.gather(*tasks)
    pred_df = pd.DataFrame(results).set_index("index")
    processed_df = df.join(pred_df, rsuffix="_complexity")
    return processed_df

In [14]:
async def analyze_df(input_df: pd.DataFrame, concurrency: int = 5) -> pd.DataFrame:
    sem = asyncio.Semaphore(concurrency)
    error_payload: dict = {
        key: None for key in list(SentenceAnalysesEvaluatorOutput.model_fields.keys())
    }

    async def _analyze_one_row(text_to_process: str):
        async with sem:
            try:
                result = await execute_sentence_analysis(text_to_process)
                result["error"] = None
                return result
            except Exception as e:
                result = {**error_payload, "error": str(e)}
                return result

    tasks = [_analyze_one_row(t) for t in input_df["text"].tolist()]
    results = await asyncio.gather(*tasks)
    llm_features_df = pd.DataFrame.from_records(results, index=input_df.index)
    llm_features_df.rename(columns={"reasoning": "grammatical_reasoning"}, inplace=True)
    processed_df = input_df.join(llm_features_df, rsuffix="_grammar")
    return processed_df

### Define the vocabulary evaluation function

In [15]:
def get_vocab_prompts_for_grade(grade: int) -> dict:
    """Return the SYSTEM_PROMPT/USER_PROMPT validated for the given grade.

    Grades 3-4 use the GRADES_3_4 prompt; all other grades (5-12) use the
    OTHER_GRADES prompt. Mirrors the standalone vocabulary evaluator.
    """
    if grade in (3, 4):
        return v_prompts.GRADE_SPECIFIC_PROMPTS["GRADES_3_4"]
    return v_prompts.GRADE_SPECIFIC_PROMPTS["OTHER_GRADES"]


def get_vocab_model_for_grade(grade: int):
    """Return the vocab model validated against the grade's prompt.

    Grades 3-4 were validated with Gemini; grades 5-12 with GPT-4.1.
    """
    if grade in (3, 4):
        return vocab_complexity_model_grades_3_4
    return vocab_complexity_model_other_grades


def normalize_vocab_output(output: dict) -> dict:
    """Convert the OTHER_GRADES integer 'answer' (1-4) into a string
    'complexity_score' so downstream normalize_label() works for all grades.
    GRADES_3_4 already returns 'complexity_score' as a string.
    """
    mapping = {
        1: "slightly complex",
        2: "moderately complex",
        3: "very complex",
        4: "exceedingly complex",
    }
    if "answer" in output and "complexity_score" not in output:
        value = output["answer"]
        if isinstance(value, str) and value.strip().isdigit():
            value = int(value)
        output["complexity_score"] = mapping.get(value, value)
    return output


async def predict_text_complexity_level(text, grade):
    """
    Predict the text complexity level as well as the complex words and reasoning.
    """

    dataset = await prepare_text_for_complexity_prediction(text, grade)

    # Select the prompt and model validated for this grade band.
    grade_prompts = get_vocab_prompts_for_grade(grade)
    messages = [
        SystemMessage(content=grade_prompts["SYSTEM_PROMPT"]),
        HumanMessagePromptTemplate.from_template(grade_prompts["USER_PROMPT"]),
    ]

    # Prepare chat prompt
    prompt = ChatPromptTemplate(
        messages,
        input_variables=prompt_vars["inputVars"],
        partial_variables={
            "format_instructions": prompt_vars["outputParser"].get_format_instructions()
        },
    )
    # Invoke the chain
    chain = prompt | get_vocab_model_for_grade(grade) | JsonOutputParser()

    # return output
    output = await chain.ainvoke(dataset)

    return normalize_vocab_output(output)

In [16]:
async def vocabulary_df(input_df: pd.DataFrame, concurrency: int = 5) -> pd.DataFrame:
    sem = asyncio.Semaphore(concurrency)
    error_payload: dict = {key: None for key in list(VocabOutput.model_fields.keys())}

    async def _vocab_one_row(text: str, grade: int):
        async with sem:
            try:
                result = await predict_text_complexity_level(text, grade)
                result["complexity_score"] = normalize_label(
                    result.get("complexity_score")
                )
                result["error"] = None
                return result
            except Exception as e:
                result = {**error_payload, "error": str(e)}
                return result

    tasks = [
        _vocab_one_row(text, grade)
        for text, grade in zip(input_df["text"], input_df["grade"])
    ]
    results = await asyncio.gather(*tasks)
    results_df = pd.DataFrame.from_records(results, index=input_df.index)
    results_df.rename(columns={"complexity_score": "vocabulary_score"}, inplace=True)
    results_df.rename(columns={"reasoning": "vocabulary_reasoning"}, inplace=True)
    processed_df = input_df.join(results_df, rsuffix="_vocab")
    return processed_df

In [17]:
async def smk_df(input_df: pd.DataFrame, concurrency: int = 5) -> pd.DataFrame:
    sem = asyncio.Semaphore(concurrency)
    error_payload: dict = {key: None for key in list(SmkOutput.model_fields.keys())}

    async def _smk_one_row(text: str, grade: int):
        async with sem:
            try:
                dataset = {
                    "text": text,
                    "grade": grade,
                    "fk_score": calculate_fk_score(text),
                }
                messages = [
                    SystemMessage(content=smk_prompts.smk_system_prompt),
                    HumanMessagePromptTemplate.from_template(smk_prompts.smk_user_prompt),
                ]
                prompt = ChatPromptTemplate(
                    messages,
                    input_variables=smk_prompt_vars["inputVars"],
                    partial_variables={
                        "format_instructions": smk_prompt_vars["outputParser"].get_format_instructions()
                    },
                )
                chain = prompt | smk_model | JsonOutputParser()
                result = await chain.ainvoke(dataset)
                score = result.get("complexity_score", "")
                result["complexity_score"] = normalize_label(
                    score.replace("_", " ") if score else score
                )
                result["error"] = None
                return result
            except Exception as e:
                result = {**error_payload, "error": str(e)}
                return result

    tasks = [
        _smk_one_row(text, grade)
        for text, grade in zip(input_df["text"], input_df["grade"])
    ]
    results = await asyncio.gather(*tasks)
    results_df = pd.DataFrame.from_records(results, index=input_df.index)
    results_df.rename(columns={"complexity_score": "smk_score"}, inplace=True)
    results_df.rename(columns={"reasoning": "smk_reasoning"}, inplace=True)
    processed_df = input_df.join(results_df, rsuffix="_smk")
    return processed_df

In [18]:
async def conventionality_df(input_df: pd.DataFrame, concurrency: int = 5) -> pd.DataFrame:
    sem = asyncio.Semaphore(concurrency)
    error_payload: dict = {key: None for key in list(ConventionalityOutput.model_fields.keys())}

    async def _conv_one_row(text: str, grade: int):
        async with sem:
            try:
                dataset = {
                    "text": text,
                    "grade": grade,
                    "fk_score": calculate_fk_score(text),
                }
                messages = [
                    SystemMessage(content=conv_prompts.conventionality_system_prompt),
                    HumanMessagePromptTemplate.from_template(conv_prompts.conventionality_user_prompt),
                ]
                prompt = ChatPromptTemplate(
                    messages,
                    input_variables=conv_prompt_vars["inputVars"],
                    partial_variables={
                        "format_instructions": conv_prompt_vars["outputParser"].get_format_instructions()
                    },
                )
                chain = prompt | conv_model | JsonOutputParser()
                result = await chain.ainvoke(dataset)
                score = result.get("complexity_score", "")
                result["complexity_score"] = normalize_label(
                    score.replace("_", " ") if score else score
                )
                result["error"] = None
                return result
            except Exception as e:
                result = {**error_payload, "error": str(e)}
                return result

    tasks = [
        _conv_one_row(text, grade)
        for text, grade in zip(input_df["text"], input_df["grade"])
    ]
    results = await asyncio.gather(*tasks)
    results_df = pd.DataFrame.from_records(results, index=input_df.index)
    results_df.rename(columns={"complexity_score": "conventionality_score"}, inplace=True)
    results_df.rename(columns={"reasoning": "conventionality_reasoning"}, inplace=True)
    processed_df = input_df.join(results_df, rsuffix="_conv")
    return processed_df

In [19]:
async def predict_text_complexity_combo_level(text: str, grade: int):
    input = {"grade": grade, "text": text}
    # Convert to a dataframe for feature engineering and processing
    input_df = pd.DataFrame.from_records([input])
    vocab_df = await vocabulary_df(input_df)
    smk_result_df = await smk_df(vocab_df)
    conv_result_df = await conventionality_df(smk_result_df)
    processed_df = await analyze_df(conv_result_df)
    final_features_df = add_engineered_features(processed_df)
    predictions_df = await evaluate_df(final_features_df)
    return predictions_df

# Test out examples

In [20]:
# Add your text & the grade level you want to evaluate for vocabulary complexity

# Clear ID = 2204
text = """
Polo went on a 24-year trip to China with his father and uncle during the Mongol Dynasty. He left Venice at the age of 17 on a boat that went through the Mediterranean Sea, Ayas, Tabriz and Kerman. Then he travelled across Asia getting as far as Beijing. On the way there he had to go over mountains and through terrible deserts, across hot burning lands and places where the cold was horrible. He served in Kublai Khan's court for 17 years. He left the Far East and returned to Venice by sea. There was sickness on board and 600 passengers and crew died and some say pirates attacked. Nevertheless, Marco Polo survived it all.
Some scholars believe that while Marco Polo did go to China, he did not go to all of the other places described in his book. He brought noodles back from China and the Italians came up with different sizes and shapes and called it pasta. Polo returned to Venice with treasures like ivory, jade, jewels, porcelain and silk.
His father had borrowed money and bought a ship. He became wealthy because of his trading in the near East.
"""

grade_level = 3

results = await predict_text_complexity_combo_level(text, grade_level)
pretty = results.iloc[0][
    [
        "vocabulary_score",
        "smk_score",
        "conventionality_score",
        "complexity_answer",
        "vocabulary_reasoning",
        "smk_reasoning",
        "conventionality_reasoning",
        "complexity_reasoning",
    ]
].to_dict()
print(json.dumps(pretty, indent=2))

{
  "vocabulary_score": "Very Complex",
  "smk_score": "Moderately Complex",
  "conventionality_score": "Slightly Complex",
  "complexity_answer": "Moderately Complex",
  "vocabulary_reasoning": "The vocabulary is very complex for a 3rd grader due to a high density of unfamiliar proper nouns and Tier 3 words presented without any contextual support. The student is confronted with a large number of new concepts in a short space, including 'Mongol Dynasty', 'Kublai Khan', and specific, likely unknown place names like 'Ayas', 'Tabriz', and 'Kerman'. This creates a significant conceptual load. Furthermore, Tier 3 vocabulary related to trade goods ('ivory', 'jade', 'porcelain') is introduced without definitions, requiring outside knowledge. While the core narrative of a trip might be understandable, the density of these specific, unsupported terms will significantly slow down comprehension and make understanding the details of the text very challenging, justifying a 'very complex' rating.",

You can copy or edit the above cell to test out different texts and grade levels.

In [21]:
full_record = results.iloc[0].to_dict()
print(json.dumps(full_record, indent=2))

{
  "grade": 3,
  "text": "\nPolo went on a 24-year trip to China with his father and uncle during the Mongol Dynasty. He left Venice at the age of 17 on a boat that went through the Mediterranean Sea, Ayas, Tabriz and Kerman. Then he travelled across Asia getting as far as Beijing. On the way there he had to go over mountains and through terrible deserts, across hot burning lands and places where the cold was horrible. He served in Kublai Khan's court for 17 years. He left the Far East and returned to Venice by sea. There was sickness on board and 600 passengers and crew died and some say pirates attacked. Nevertheless, Marco Polo survived it all.\nSome scholars believe that while Marco Polo did go to China, he did not go to all of the other places described in his book. He brought noodles back from China and the Italians came up with different sizes and shapes and called it pasta. Polo returned to Venice with treasures like ivory, jade, jewels, porcelain and silk.\nHis father had bor